Link data: https://www.kaggle.com/datasets/miguelcorraljr/ted-ultimate-dataset/data

Data downloading, preprocessing

In [4]:
!pip install numpy
!pip install pandas
!pip install nltk unidecode
!pip install opencc

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import string # used for preprocessing
import re # used for preprocessing
import nltk
import numpy as np
import math
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords # used for preprocessing
from nltk.stem import WordNetLemmatizer # used for preprocessing
from collections import defaultdict, Counter
from nltk.util import ngrams
from nltk.corpus import reuters
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import Laplace
from unidecode import unidecode
import opencc

# Initialize OpenCC for Traditional to Simplified Chinese conversion
converter = opencc.OpenCC('t2s')  # Converts Traditional to Simplified


nltk.download('punkt')
nltk.download('reuters')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

import os
os.environ['KAGGLE_USERNAME'] = "chillyon"
os.environ['KAGGLE_KEY'] = "a1af1706a9493950a19751152069f960"
from kaggle.api.kaggle_api_extended import KaggleApi

# Download and Load the dataset from Kaggle
api = KaggleApi()
api.authenticate()
dataset_name = 'miguelcorraljr/ted-ultimate-dataset'
api.dataset_download_files(dataset_name, path='./', unzip=True)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Dataset URL: https://www.kaggle.com/datasets/miguelcorraljr/ted-ultimate-dataset


Get the data

In [6]:
os.listdir('./2020-05-01')
# Load Titanic dataset from Kaggle
# lang_ver = ['en']
train_data = pd.read_csv(f"./2020-05-01/ted_talks_en.csv")

Change to column

In [9]:
df = pd.DataFrame(train_data)
print("\nData converted to DataFrame:")
# Select the transcription column
df = df[['transcript']]
# Clean the null in the data
df = df.dropna(subset=['transcript'])
print(df.head())


Data converted to DataFrame:
                                          transcript
0  Thank you so much, Chris. And it's truly a gre...
1  About 10 years ago, I took on the task to teac...
2  (Music: "The Sound of Silence," Simon & Garfun...
3  If you're here today — and I'm very happy that...
4  Good morning. How are you? (Audience) Good. It...


Preprocessing text function

In [10]:
# def clean_text(text):
#     # Replace characters not matching A-Za-z0-9(),!?' with a space
#     return re.sub(r"[^A-Za-z0-9(),!?'`]", " ", text)

def text_lowercase(text):
    return text.lower()
def remove_punctuation(text):
    # Repair common punctuation
    text = re.sub(r",", ",", text)  # No change for comma
    text = re.sub(r"!", "!", text)  # No change for exclamation
    text = re.sub(r"\(", "(", text)  # No change for left parenthesis
    text = re.sub(r"\)", ")", text)  # No change for right parenthesis
    text = re.sub(r"\?", "?", text)  # No change for question mark
    return text
# Load and preprocess data
def preprocessing(text):
    lower_text = text_lowercase(text)
    punct_text = remove_punctuation(lower_text)
    return punct_text

In [11]:
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = text.strip()
    return text

Preprocessing text in data

In [13]:
df['cleaned_transcript'] = df['transcript'].apply(clean_text)

df = df[['cleaned_transcript']]        # Only keep cleaned transcript
print(f"Language en:")
print(df.head())

Language en:
                                  cleaned_transcript
0  thank you so much chris and its truly a great ...
1  about  years ago i took on the task to teach g...
2  music the sound of silence simon  garfunkel he...
3  if youre here today  and im very happy that yo...
4  good morning how are you audience good its bee...


Vectorizing

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Split data into features and target variable
X = df['cleaned_transcript']

# Vectorizing text using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))  # Consider unigrams and bigrams
X_vectorized = vectorizer.fit_transform(X)

# Convert to dense array
X_dense = X_vectorized.toarray()

# Create DataFrame with words as columns
df_tfidf = pd.DataFrame(X_dense, columns=vectorizer.get_feature_names_out())

In [22]:
print(X_vectorized.shape)
print(df_tfidf.shape)
df_tfidf['target'] = None

# Display a sample
print(df_tfidf.head())


(4005, 5000)
(4005, 5000)
    ability  ability to      able   able to     about  about all  about and  \
0  0.000000         0.0  0.000000  0.000000  0.059046   0.000000   0.000000   
1  0.000000         0.0  0.008482  0.008505  0.039461   0.000000   0.000000   
2  0.000000         0.0  0.000000  0.000000  0.024330   0.000000   0.000000   
3  0.000000         0.0  0.000000  0.000000  0.027778   0.000000   0.000000   
4  0.014681         0.0  0.000000  0.000000  0.060794   0.010936   0.010139   

   about how  about is  about it  ...  youre in  youre looking  youre not  \
0   0.000000       0.0  0.027297  ...       0.0            0.0   0.000000   
1   0.000000       0.0  0.000000  ...       0.0            0.0   0.000000   
2   0.013928       0.0  0.006093  ...       0.0            0.0   0.008841   
3   0.015901       0.0  0.006956  ...       0.0            0.0   0.000000   
4   0.006960       0.0  0.012179  ...       0.0            0.0   0.044184   

   yourself  youth  youtube     youv


Save to CSV and download them

In [23]:
df_tfidf.to_csv(f"tfidf_en.csv", index=False)
from google.colab import files
files.download("tfidf_en.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
os.listdir('./')

['.config', '2020-05-01', 'tfidf_en.csv', 'sample_data']

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model training

1. Decision Trees:

Feature transformation for text data

Handle high dimensionality

Implement pruning strategies